# Tarea 2 · Parte 1 — Corpus y extracción

**Pregunta guía:** ¿se pueden convertir documentos escaneados y entrevistas en audio en un
archivo de texto estructurado y buscable?

Alexander Noblejas (`@AlexNoblejas`) · Python Intermedio, Diplomatura QLAB

Este notebook arma el corpus, le pasa OCR a los documentos y ASR a los audios, seudonimiza,
limpia y exporta los tres CSV que consumen la Parte 2 y el dashboard.

In [ ]:
import hashlib, json, platform, subprocess, sys, time
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import torch

import corpus_audio, corpus_documentos, voz

AQUI = Path.cwd()
DATOS = AQUI / "datos"
(DATOS / "salidas").mkdir(parents=True, exist_ok=True)

DISPOSITIVO = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

import paddleocr
print(f"Fecha de extracción: {date.today().isoformat()}")
print(f"Python {sys.version.split()[0]} · {platform.system()} {platform.release()}")
print(f"PaddleOCR {paddleocr.__version__} · PyTorch {torch.__version__}")
print(f"Dispositivo: {DISPOSITIVO}" + (f" ({torch.cuda.get_device_name(0)})" if DISPOSITIVO == "cuda" else ""))

## 1. Fuentes y licencias

**Audio — 26 archivos de Wikimedia Commons, todos CC BY-SA 4.0**, que al segmentar los dos
audios largos quedan en ~37 clips y 41 minutos de español peruano. Ningún audio fue grabado
para esta tarea.

| Grupo | Qué es |
|---|---|
| `calle` | Entrevistas de calle de Wikimedia Perú y colaboradores (2022) |
| `conferencia` | Dirigente de Fenmucarinap en conferencia de prensa |
| `testimonio` | Testimonio de Lucinda Quispealaya, Junín (8,8 min, se segmenta) |
| `resonando` | Proyecto *Resonando Juntas* (WikiAcción Perú + Asimtría): testimonios ya anónimos |
| `ruido` | Consignas coreadas en marcha: multitud y música — los casos difíciles |

**Documentos — 25.** Los 8 impresos nítidos son páginas de resoluciones oficiales del CEPLAN:
el artículo 9 del Decreto Legislativo 822 excluye de protección "los textos oficiales de
carácter legislativo, administrativo o judicial", así que su texto puede publicarse. Las 8
fotos de celular y los 9 manuscritos son capturas propias sin datos personales reales.

El detalle por archivo, con URL, licencia y procedimiento de recuperación, queda en
`datos/fuentes.csv`.

In [ ]:
corpus_audio.listar()
print()
corpus_documentos.listar()

In [ ]:
# Si el corpus todavía no está en disco, estas dos líneas lo traen.
# Respetan las condiciones de Wikimedia y de gob.pe: User-Agent descriptivo y 1,5 s entre peticiones.
if not (DATOS / "clips.csv").exists():
    filas = corpus_audio.convertir() if (DATOS / "raw" / "audio").exists() else (
        corpus_audio.descargar() or corpus_audio.convertir())
    corpus_audio.escribir_fuentes(filas)
    corpus_audio.escribir_clips(corpus_audio.segmentar(filas))

if not list((DATOS / "raw" / "documentos").glob("impreso_*.jpg")):
    corpus_documentos.descargar()
    corpus_documentos.escribir_fuentes(corpus_documentos.renderizar())

clips = pd.read_csv(DATOS / "clips.csv")
fuentes = pd.read_csv(DATOS / "fuentes.csv")
print(f"{len(clips)} clips · {clips.duracion_seg.sum() / 60:.1f} min")
print(f"{len(fuentes)} filas en fuentes.csv · licencias: {fuentes.licencia.nunique()} distintas")

## 2. OCR de los 25 documentos (sesión 10)

Configuración base de PaddleOCR, la misma de la clase: sin clasificador de orientación de
página, sin desdoblado y sin rotación de líneas. Es deliberadamente la configuración *sin
personalizar*, porque es la línea base que la Parte 2 compara contra Tesseract.

El tipo de cada documento sale del prefijo del archivo: `impreso_`, `foto_` o `manu_`.

In [ ]:
OPCIONES = dict(
    use_doc_orientation_classify=False,   # no girar la página completa
    use_doc_unwarping=False,              # no "planchar" hojas dobladas
    use_textline_orientation=False,       # no girar cada línea
)
MODELO_OCR = f"PaddleOCR {paddleocr.__version__} (engine=transformers, lang=es, device={DISPOSITIVO})"

ocr = paddleocr.PaddleOCR(engine="transformers", lang="es", device=DISPOSITIVO, **OPCIONES)
print(MODELO_OCR)

In [ ]:
def leer(ruta, modelo=ocr) -> pd.DataFrame:
    """OCR de una imagen: una fila por línea de texto, con su confianza."""
    r = modelo.predict(str(ruta))[0]
    return pd.DataFrame({"texto": r["rec_texts"], "confianza": np.round(r["rec_scores"], 3)})


TIPOS = {"impreso": "impreso", "foto": "foto", "manu": "manuscrito"}

def tipo_de(nombre: str) -> str:
    return TIPOS[nombre.split("_")[0]]


imagenes = sorted((DATOS / "raw" / "documentos").glob("*.jpg")) + \
           sorted((DATOS / "raw" / "documentos").glob("*.jpeg")) + \
           sorted((DATOS / "raw" / "documentos").glob("*.png"))
print(f"{len(imagenes)} imágenes en datos/raw/documentos/")
pd.Series([tipo_de(p.stem) for p in imagenes]).value_counts()

In [ ]:
filas_doc, lineas_doc = [], []
for ruta in imagenes:
    t0 = time.perf_counter()
    try:
        tabla = leer(ruta)
    except Exception as e:
        print(f"  {ruta.stem}: OCR falló ({type(e).__name__}: {e})")
        tabla = pd.DataFrame({"texto": [], "confianza": []})
    segundos = time.perf_counter() - t0
    texto = "\n".join(tabla.texto.tolist())
    filas_doc.append({
        "id": ruta.stem, "archivo": f"datos/raw/documentos/{ruta.name}", "tipo": tipo_de(ruta.stem),
        "texto_ocr": texto, "modelo_ocr": MODELO_OCR, "texto_referencia": "",
        "n_lineas": len(tabla), "confianza_media": round(tabla.confianza.mean(), 3) if len(tabla) else np.nan,
        "n_caracteres": len(texto), "segundos_ocr": round(segundos, 2),
    })
    tabla.insert(0, "id", ruta.stem)
    lineas_doc.append(tabla)
    print(f"  {ruta.stem:12s} {tipo_de(ruta.stem):10s} {len(tabla):3d} líneas  "
          f"{len(texto):5d} car.  conf. {tabla.confianza.mean() if len(tabla) else float('nan'):.3f}  {segundos:5.1f} s")

documentos = pd.DataFrame(filas_doc)
pd.concat(lineas_doc).to_csv(DATOS / "salidas" / "ocr_lineas.csv", index=False)
print(f"\n{len(documentos)} documentos · {documentos.n_caracteres.sum():,} caracteres extraídos")
documentos.groupby("tipo")[["n_caracteres", "confianza_media", "segundos_ocr"]].mean().round(3)

## 3. ASR de los clips (sesión 13)

**faster-whisper `small` en CPU.** Tres decisiones que conviene justificar, porque se apartan
de la receta por defecto:

1. **CPU y no GPU.** `pip install torch` entregó la rueda `+cpu`, y reinstalar la de CUDA
   arrastra sus propias restricciones de numpy —el conflicto que ya costó caro en la Tarea 1—.
   Queda registrado como limitación técnica.
2. **Modelo `small`.** Es además la configuración que el enunciado fija como línea base en la
   Parte 2, así que la Parte 1 y la línea base usan exactamente el mismo modelo.
3. **VAD propio, por energía, en numpy** (`corpus_audio.regiones_de_voz`). El VAD de
   faster-whisper corre sobre onnxruntime y en esta máquina tumbaba el intérprete sin dejar
   traceback: fue lo que hizo fallar la segmentación en silencio. El de repuesto cubre el 100 %
   del audio cortando en pausas en trozos de hasta 25 s, y descarta un trozo solo si está en
   silencio prácticamente absoluto. Un VAD agresivo con umbral relativo se comía el 60 % del
   `clip_06`, porque en una grabación de calle el ruido de fondo levanta el umbral por encima
   de la voz; perder habla arruina el WER mucho más que mandar al modelo un par de segundos de
   silencio.

Las marcas `[HABLANTE_n]` se agregan solo donde la diarización aporta algo: el `clip_06` tiene
tres voces declaradas por la fuente (el entrevistador y dos vendedoras).

In [ ]:
from faster_whisper import WhisperModel

MODELO_ASR = "faster-whisper small (CPU, int8, VAD por energía en numpy)"
t0 = time.perf_counter()
asr = WhisperModel("small", device="cpu", compute_type="int8")
print(f"{MODELO_ASR} · cargado en {time.perf_counter() - t0:.1f} s")

In [ ]:
def transcribir_clip(audio) -> list[voz.Segmento]:
    """Transcribe un clip trozo por trozo y devuelve los segmentos con tiempos absolutos."""
    segmentos = []
    for a, b in corpus_audio.regiones_de_voz(audio):
        trozo = audio[int(a * voz.SR):int(b * voz.SR)]
        segs, _info = asr.transcribe(trozo, language="es", beam_size=5, vad_filter=False)
        for s in segs:
            texto = s.text.strip()
            if texto:
                segmentos.append(voz.Segmento(round(a + s.start, 2), round(a + s.end, 2), texto))
    return segmentos


MULTIHABLANTE = {"clip_06"}          # el único clip con más de dos voces declaradas

filas_asr = []
for fila in clips.itertuples():
    audio = voz.cargar_audio(AQUI / fila.archivo_audio)
    t0 = time.perf_counter()
    segmentos = transcribir_clip(audio)
    segundos = time.perf_counter() - t0

    n_hablantes = 1
    if fila.fuente_id in MULTIHABLANTE and segmentos:
        tramos = [(s.inicio, s.fin) for s in segmentos]
        etiquetas = corpus_audio.diarizar_simple(audio, tramos,
                                                 n_hablantes=int(fila.hablantes_declarados))
        segmentos = [voz.Segmento(s.inicio, s.fin, s.texto, f"H{e + 1}")
                     for s, e in zip(segmentos, etiquetas)]
        n_hablantes = len({s.hablante for s in segmentos})

    texto = voz.a_texto(segmentos)
    (DATOS / "salidas" / f"{fila.clip_id}.srt").write_text(voz.a_srt(segmentos), encoding="utf-8")
    (DATOS / "salidas" / f"{fila.clip_id}.txt").write_text(texto, encoding="utf-8")

    filas_asr.append({
        "id": fila.clip_id, "archivo_audio": fila.archivo_audio, "duracion_seg": fila.duracion_seg,
        "idioma": "es", "texto_asr": texto, "modelo_asr": MODELO_ASR, "n_hablantes": n_hablantes,
        "texto_referencia": "", "fuente_id": fila.fuente_id, "grupo": fila.grupo,
        "n_palabras": len(texto.split()), "segundos_asr": round(segundos, 2),
        "factor_tiempo_real": round(segundos / fila.duracion_seg, 3),
    })
    print(f"  {fila.clip_id:14s} {fila.duracion_seg:6.1f} s  {len(texto.split()):5d} palabras  "
          f"x{segundos / fila.duracion_seg:.2f} tiempo real", flush=True)

transcripciones = pd.DataFrame(filas_asr)
print(f"\n{len(transcripciones)} clips · {transcripciones.duracion_seg.sum() / 60:.1f} min · "
      f"{transcripciones.n_palabras.sum():,} palabras")
transcripciones.groupby("grupo").agg(
    clips=("id", "size"), segundos=("duracion_seg", "sum"), palabras=("n_palabras", "sum"),
    factor_medio=("factor_tiempo_real", "mean")).round(2)

In [ ]:
# La diarización, mirada de cerca: quién dice qué en el clip con tres voces
cid = sorted(MULTIHABLANTE)[0]
fila = clips[clips.fuente_id == cid].iloc[0]
audio = voz.cargar_audio(AQUI / fila.archivo_audio)
segs = transcribir_clip(audio)
etiquetas = corpus_audio.diarizar_simple(audio, [(s.inicio, s.fin) for s in segs],
                                         n_hablantes=int(fila.hablantes_declarados))
segs = [voz.Segmento(s.inicio, s.fin, s.texto, f"H{e + 1}") for s, e in zip(segs, etiquetas)]
print(f"{fila.clip_id}: {len({s.hablante for s in segs})} hablantes sobre {len(segs)} turnos")
print("La huella es MFCC, no un modelo de voces: separa turnos, no afirma identidades.")
voz.a_tabla(segs).head(12)

## 4. Seudonimización

Las entrevistas de calle nombran a personas reales, y el enunciado prohíbe publicar cualquier
dato personal. `voz.seudonimizar` compara de forma difusa, así que atrapa el nombre aunque el
modelo lo haya escrito mal ("Isabel Vázquez" por "Isabel Vásquez").

**El reemplazo automático no basta y el enunciado lo dice.** Después de esta celda hay que leer
los textos y revisar que no quede ningún nombre, dirección ni número de documento.

In [ ]:
REEMPLAZOS = {
    # nombres que aparecen en las descripciones de Commons de las entrevistas de calle
    "Elena Duran": "[PERSONA_1]", "Isabel Vásquez": "[PERSONA_2]", "Elsa Barrientos": "[PERSONA_3]",
    "Aurora Coronado": "[PERSONA_4]", "Lucinda Quispealaya": "[PERSONA_5]",
}

def seudonimizar_serie(s: pd.Series) -> tuple[pd.Series, int]:
    salida, cambios = [], 0
    for texto in s.fillna(""):
        nuevo = voz.seudonimizar(texto, REEMPLAZOS)
        cambios += sum(nuevo.count(v) - texto.count(v) for v in REEMPLAZOS.values())
        salida.append(nuevo)
    return pd.Series(salida, index=s.index), cambios

transcripciones["texto_asr"], n_audio = seudonimizar_serie(transcripciones.texto_asr)
documentos["texto_ocr"], n_doc = seudonimizar_serie(documentos.texto_ocr)
print(f"Entidades reemplazadas: {n_audio} en las transcripciones y {n_doc} en los textos OCR")

# Red de seguridad: ¿quedó algo que parezca un DNI (8 dígitos seguidos) o un correo?
import re
sospechoso = re.compile(r"\b\d{8}\b|@[\w.-]+\.\w+")
for nombre, df, col in (("audio", transcripciones, "texto_asr"), ("documentos", documentos, "texto_ocr")):
    hits = df[df[col].str.contains(sospechoso, na=False)]
    print(f"  {nombre}: {len(hits)} textos con patrón sospechoso" + (f" -> {hits.id.tolist()}" if len(hits) else ""))

## 5. Limpieza

Duplicados por hash del archivo, documentos ilegibles (OCR que no devolvió texto) y audios sin
voz (transcripción vacía). Cada descarte queda reportado con su motivo.

In [ ]:
def md5(ruta: Path) -> str:
    return hashlib.md5(Path(ruta).read_bytes()).hexdigest()

descartes = []

documentos["hash"] = [md5(AQUI / a) for a in documentos.archivo]
dup_doc = documentos[documentos.duplicated("hash", keep="first")]
for r in dup_doc.itertuples():
    descartes.append({"id": r.id, "modalidad": "documento", "motivo": "archivo duplicado (mismo hash)"})

ilegibles = documentos[(documentos.n_caracteres < 20) & ~documentos.id.isin(dup_doc.id)]
for r in ilegibles.itertuples():
    descartes.append({"id": r.id, "modalidad": "documento",
                      "motivo": f"ilegible: OCR devolvió {r.n_caracteres} caracteres"})

sin_voz = transcripciones[transcripciones.n_palabras < 3]
for r in sin_voz.itertuples():
    descartes.append({"id": r.id, "modalidad": "audio", "motivo": f"sin voz: {r.n_palabras} palabras"})

documentos = documentos[~documentos.id.isin([d["id"] for d in descartes])].reset_index(drop=True)
transcripciones = transcripciones[~transcripciones.id.isin([d["id"] for d in descartes])].reset_index(drop=True)

tabla_descartes = pd.DataFrame(descartes, columns=["id", "modalidad", "motivo"])
tabla_descartes.to_csv(DATOS / "salidas" / "descartes.csv", index=False)
print(f"{len(tabla_descartes)} descartes · quedan {len(documentos)} documentos y {len(transcripciones)} clips "
      f"({transcripciones.duracion_seg.sum() / 60:.1f} min)")
tabla_descartes

In [ ]:
# Control de mínimos del enunciado: 25 documentos (3 tipos) y 30 min en 20+ clips
conteo = documentos.tipo.value_counts()
print("Documentos por tipo:", dict(conteo))
assert len(documentos) >= 25, f"faltan documentos: {len(documentos)} de 25"
assert {"impreso", "foto", "manuscrito"} <= set(conteo.index), "falta algún tipo de documento"
assert len(transcripciones) >= 20, f"faltan clips: {len(transcripciones)} de 20"
assert transcripciones.duracion_seg.sum() >= 30 * 60, f"faltan minutos: {transcripciones.duracion_seg.sum() / 60:.1f}"
assert (transcripciones.n_hablantes > 1).any(), "ningún clip con más de un hablante"
print("Mínimos del enunciado: cumplidos")

## 6. Análisis exploratorio

Tres gráficos, cada uno titulado con la conclusión y no con la variable.

In [ ]:
orden = ["impreso", "foto", "manuscrito"]
g = documentos.tipo.value_counts().reindex(orden).reset_index()
g.columns = ["tipo", "n"]
fig = px.bar(g, x="tipo", y="n", text="n", color="tipo",
             title="Los manuscritos son el grupo más numeroso del corpus: son el caso difícil que el pipeline tiene que resistir",
             labels={"tipo": "Tipo de documento", "n": "Documentos"})
fig.update_layout(showlegend=False, title_font_size=13)
fig.show()

In [ ]:
fig = px.bar(transcripciones.sort_values("duracion_seg"), x="id", y="duracion_seg", color="grupo",
             title=f"Segmentar los dos audios largos deja {len(transcripciones)} clips parejos de ~40 a 90 s, "
                   f"y ninguno domina los {transcripciones.duracion_seg.sum() / 60:.0f} minutos del corpus",
             labels={"id": "Clip", "duracion_seg": "Duración (s)", "grupo": "Grupo"})
fig.update_layout(title_font_size=13, xaxis_tickangle=-60)
fig.show()

In [ ]:
largos = pd.concat([
    documentos.assign(modalidad="Documentos (caracteres OCR)", valor=documentos.n_caracteres)[["modalidad", "valor"]],
    transcripciones.assign(modalidad="Audio (palabras ASR)", valor=transcripciones.n_palabras)[["modalidad", "valor"]],
])
fig = px.histogram(largos, x="valor", facet_col="modalidad", nbins=20, color="modalidad",
                   title="El texto por documento se reparte muy desigual, mientras el de los clips se concentra: "
                         "la variabilidad del OCR es la que manda en el esfuerzo de revisión",
                   labels={"valor": "Longitud del texto extraído"})
fig.update_xaxes(matches=None)
fig.update_layout(showlegend=False, title_font_size=13)
fig.show()

## 7. Validación

Los extremos de cada modalidad, para comprobar que los histogramas no esconden un error: un
documento con muchísimo texto suele ser una página densa, y uno con muy poco, un manuscrito
que el OCR no supo leer.

In [ ]:
for etiqueta, df, col in (("documentos (caracteres OCR)", documentos, "n_caracteres"),
                          ("clips (palabras ASR)", transcripciones, "n_palabras")):
    print(f"\n=== 3 {etiqueta} más largos ===")
    print(df.nlargest(3, col)[["id", col]].to_string(index=False))
    print(f"=== 3 {etiqueta} más cortos ===")
    print(df.nsmallest(3, col)[["id", col]].to_string(index=False))

In [ ]:
# Un ojo encima del texto real, no solo de los conteos
print("--- documento con más texto ---")
print(documentos.nlargest(1, "n_caracteres").texto_ocr.iloc[0][:400], "…\n")
print("--- documento con menos texto ---")
print(repr(documentos.nsmallest(1, "n_caracteres").texto_ocr.iloc[0][:400]), "\n")
print("--- clip con más palabras ---")
print(transcripciones.nlargest(1, "n_palabras").texto_asr.iloc[0][:400], "…")

## 8. Exportación

`texto_referencia` sale vacío a propósito: se llena en la Parte 2, a mano, sobre los 10
documentos y 10 clips de la muestra de evaluación.

In [ ]:
COLS_DOC = ["id", "archivo", "tipo", "fuente", "texto_ocr", "modelo_ocr", "texto_referencia"]
COLS_ASR = ["id", "archivo_audio", "duracion_seg", "idioma", "texto_asr", "modelo_asr",
            "n_hablantes", "texto_referencia"]

fuente_por_id = dict(zip(fuentes.id, fuentes.titulo_fuente + " — " + fuentes.licencia))
documentos["fuente"] = [
    fuente_por_id.get(i, "Captura propia del estudiante, sin datos personales reales")
    for i in documentos.id
]

documentos.to_csv(DATOS / "documentos.csv", index=False, columns=COLS_DOC + [
    "n_lineas", "confianza_media", "n_caracteres", "segundos_ocr"])
transcripciones.to_csv(DATOS / "transcripciones.csv", index=False, columns=COLS_ASR + [
    "fuente_id", "grupo", "n_palabras", "segundos_asr", "factor_tiempo_real"])

print("documentos.csv     ", pd.read_csv(DATOS / "documentos.csv").shape)
print("transcripciones.csv", pd.read_csv(DATOS / "transcripciones.csv").shape)
print("fuentes.csv        ", fuentes.shape)

In [ ]:
pd.read_csv(DATOS / "documentos.csv")[COLS_DOC].head()

In [ ]:
pd.read_csv(DATOS / "transcripciones.csv")[COLS_ASR].head()

---

**Cierre de la Parte 1.** El corpus está en disco y documentado, los dos CSV salen con las
columnas que pide el enunciado, y `texto_referencia` queda listo para la transcripción manual
de la Parte 2.